# 04 — GNN Model: GraphSAGE Node Classifier

## Approach

Each BGT tile becomes a **graph**:
- **Nodes** = BGT polygons
- **Edges** = spatial adjacency (polygons within `adjacency_distance_m` of each other)
- **Node features** = geometric + semantic attributes (area, perimeter, compactness, class encoding, etc.)
- **Node labels** = `survived` (1) or `dropped` (0), computed in notebook 01 by IoU matching

The **GraphSAGE** model learns to predict survival for each node by aggregating
information from its neighbours. This is a node-level binary classification task.

### Why GraphSAGE over GCN?
- Works on graphs of varying size (different tiles have different polygon counts)
- Inductive — generalizes to unseen cities without retraining
- Computationally efficient for large graphs

### No rasterization
All operations are on the original vector geometries. No information is lost
to pixel discretization.

Run **01_data_prep.ipynb** first.


## 0. Install dependencies (Colab)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Install PyTorch Geometric — version must match your torch install
    import torch
    torch_version  = torch.__version__.split('+')[0]
    cuda_version   = 'cu118' if torch.cuda.is_available() else 'cpu'
    !pip install torch-scatter torch-sparse torch-geometric \
        -f https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html --quiet
else:
    # Local: install via pip
    # pip install torch-geometric
    pass

## 1. CONFIG

In [1]:
from pathlib import Path

CONFIG = {
    "output_root":  Path("processed"),
    "model_dir":    Path("models"),
    "crs":          "EPSG:28992",

    # ── Class column (must match notebook 01) ─────────────
    "bgt_class_col": "plus-fysiekVoorkomenWegdeel",

    # ── Graph construction ────────────────────────────────
    # Two polygons get an edge if they are within this distance (metres)
    "adjacency_distance_m": 0.5,

    # ── Node features ─────────────────────────────────────
    # These are computed per polygon and used as GNN input.
    # All are normalised to [0,1] or z-scored before training.
    # Options: area, perimeter, compactness, aspect_ratio,
    #          n_vertices, class_encoded, n_neighbors
    "node_features": [
        "area",
        "perimeter",
        "compactness",
        "aspect_ratio",
        "n_vertices",
        "class_encoded",
    ],

    # ── Model architecture ────────────────────────────────
    # Hidden dimension of each GraphSAGE layer
    "hidden_dim":   64,
    # Number of GraphSAGE message-passing layers
    # More layers = larger receptive field (further neighbours considered)
    "n_layers":     3,
    # Dropout probability
    "dropout":      0.3,

    # ── Training ──────────────────────────────────────────
    "num_epochs":          50,
    "learning_rate":       1e-3,
    "weight_decay":        1e-4,
    "early_stop_patience": 8,

    # Class imbalance: if many more dropped than survived polygons,
    # weight the loss to penalise missed survivals more.
    # Set to None to compute automatically from training data.
    "pos_weight":   None,

    "seed": 42,
}

CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)
print("CONFIG loaded.")

CONFIG loaded.


In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.12.0+cpu
None


## 2. Imports

In [3]:
import json
import random
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn   import SAGEConv

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'torch_geometric'

## 3. Load encoder

In [ ]:
def load_encoder(path):
    """Load label encoder from JSON — no pickle needed."""
    with open(path) as f:
        data = json.load(f)
    class Enc: pass
    enc = Enc()
    enc.unknown_label = data["unknown_label"]
    enc.classes_      = data["classes"]
    enc.label_to_int  = data["label_to_int"]
    enc.int_to_label  = {int(k): v for k, v in data["int_to_label"].items()}
    return enc

bgt_encoder = load_encoder(CONFIG["output_root"] / "bgt_encoder.json")
N_CLASSES   = len(bgt_encoder.classes_)
N_FEATURES  = len(CONFIG["node_features"])

print(f"BGT classes  : {N_CLASSES}")
print(f"Node features: {N_FEATURES}  {CONFIG['node_features']}")
assert N_CLASSES > 0, "BGT encoder is empty — re-run notebook 01"

## 4. Graph construction

Convert one GeoDataFrame tile into a PyTorch Geometric `Data` object.

In [ ]:
def compute_node_features(gdf: gpd.GeoDataFrame,
                           feature_names: list,
                           class_col_encoded: str) -> np.ndarray:
    """
    Compute a feature matrix of shape (N, n_features).
    All geometric features are log-transformed then z-scored
    to bring them into a similar range for the GNN.
    """
    df = pd.DataFrame(index=gdf.index)

    # Geometric features
    area      = gdf.geometry.area
    perimeter = gdf.geometry.length

    df["area"]        = np.log1p(area)
    df["perimeter"]   = np.log1p(perimeter)
    df["compactness"] = (4 * np.pi * area) / (perimeter ** 2 + 1e-9)

    bounds = gdf.geometry.bounds
    w = (bounds["maxx"] - bounds["minx"]).clip(lower=1e-6)
    h = (bounds["maxy"] - bounds["miny"]).clip(lower=1e-6)
    df["aspect_ratio"] = np.log1p(w / h)

    df["n_vertices"] = np.log1p(gdf.geometry.apply(
        lambda g: len(g.exterior.coords) if g.geom_type == "Polygon"
        else sum(len(p.exterior.coords) for p in g.geoms)
    ))

    # Class encoding — normalised to [0, 1]
    if class_col_encoded in gdf.columns:
        df["class_encoded"] = gdf[class_col_encoded] / max(N_CLASSES - 1, 1)
    else:
        df["class_encoded"] = 0.0

    # Select only requested features, in order
    matrix = df[feature_names].fillna(0.0).values.astype(np.float32)

    # Z-score normalise each feature column
    mean = matrix.mean(axis=0)
    std  = matrix.std(axis=0) + 1e-8
    return (matrix - mean) / std


def build_adjacency(gdf: gpd.GeoDataFrame, dist: float):
    """
    Build edge_index (COO format) for a spatial adjacency graph.
    Two polygons are connected if they are within `dist` metres.
    Returns a (2, E) LongTensor of undirected edges.
    """
    if len(gdf) == 0:
        return torch.zeros((2, 0), dtype=torch.long)

    # Buffer polygons slightly then find intersections
    buffered = gdf.copy()
    buffered["geometry"] = gdf.geometry.buffer(dist)

    joined = gpd.sjoin(
        buffered[["geometry"]].reset_index().rename(columns={"index": "src"}),
        gdf[["geometry"]].reset_index().rename(columns={"index": "dst"}),
        how="inner", predicate="intersects"
    )

    # Remove self-loops and keep only unique undirected edges
    edges = joined[["src", "dst"]].values
    edges = edges[edges[:, 0] != edges[:, 1]]
    # Sort each edge so (i,j) and (j,i) become the same
    edges = np.sort(edges, axis=1)
    edges = np.unique(edges, axis=0)
    # Make bidirectional for message passing
    edges_bi = np.concatenate([edges, edges[:, ::-1]], axis=0)

    return torch.tensor(edges_bi.T, dtype=torch.long)


def tile_to_graph(bgt: gpd.GeoDataFrame, config: dict) -> Data:
    """
    Convert one BGT tile GeoDataFrame to a PyTorch Geometric Data object.

    Returns Data with:
      x          : (N, n_features) float node feature matrix
      edge_index : (2, E) long adjacency
      y          : (N,) long survival labels (0/1)
      n_nodes    : int
    """
    bgt = bgt.reset_index(drop=True)

    bgt_col_enc = f"{config['bgt_class_col']}_encoded"

    # Node features
    x = compute_node_features(bgt, config["node_features"], bgt_col_enc)
    x = torch.tensor(x, dtype=torch.float)

    # Edges
    edge_index = build_adjacency(bgt, config["adjacency_distance_m"])

    # Labels
    if "survived" in bgt.columns:
        y = torch.tensor(bgt["survived"].fillna(0).values, dtype=torch.long)
    else:
        y = torch.zeros(len(bgt), dtype=torch.long)

    return Data(x=x, edge_index=edge_index, y=y, n_nodes=len(bgt))


# Quick test on one tile
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

sample_bgt = gpd.read_file(tile_index["train"][0]["bgt"])
sample_graph = tile_to_graph(sample_bgt, CONFIG)

print(f"Sample graph:")
print(f"  Nodes      : {sample_graph.x.shape[0]}")
print(f"  Features   : {sample_graph.x.shape[1]}")
print(f"  Edges      : {sample_graph.edge_index.shape[1] // 2} undirected")
print(f"  Label dist : {sample_graph.y.sum().item()} survived / {sample_graph.y.shape[0]} total")

## 5. GraphSAGE model

In [ ]:
class GraphSAGE(nn.Module):
    """
    GraphSAGE node classifier for BGT polygon survival prediction.

    Architecture:
      - n_layers of SAGEConv (message passing)
      - BatchNorm + ReLU + Dropout after each layer
      - Final linear head → 2 logits (dropped / survived)

    Each SAGEConv layer aggregates features from 1-hop neighbours.
    With n_layers=3, each node sees up to 3-hop neighbours.
    """

    def __init__(self, in_dim: int, hidden_dim: int, n_layers: int, dropout: float):
        super().__init__()

        self.convs  = nn.ModuleList()
        self.norms  = nn.ModuleList()
        self.dropout = dropout

        # First layer
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Intermediate layers
        for _ in range(n_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Output head: hidden_dim → 2 (binary classification)
        self.head = nn.Linear(hidden_dim, 2)

    def forward(self, x, edge_index):
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)   # returns raw logits, shape (N, 2)


model = GraphSAGE(
    in_dim     = N_FEATURES,
    hidden_dim = CONFIG["hidden_dim"],
    n_layers   = CONFIG["n_layers"],
    dropout    = CONFIG["dropout"],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GraphSAGE parameters: {total_params:,}")

# Quick forward pass check
with torch.no_grad():
    g = sample_graph.to(DEVICE)
    out = model(g.x, g.edge_index)
    print(f"Output shape: {out.shape}  — expected ({g.x.shape[0]}, 2)")

## 6. Class weight (handles imbalanced survival labels)

In [ ]:
if CONFIG["pos_weight"] is None:
    # Estimate from a sample of training tiles
    survived_count = 0
    dropped_count  = 0
    for tile in tile_index["train"][:100]:  # sample first 100
        bgt = gpd.read_file(tile["bgt"])
        if "survived" in bgt.columns:
            survived_count += bgt["survived"].sum()
            dropped_count  += (1 - bgt["survived"]).sum()

    if survived_count > 0:
        pos_weight = dropped_count / survived_count
    else:
        pos_weight = 1.0

    print(f"Auto pos_weight: {pos_weight:.2f}  "
          f"(survived={survived_count}, dropped={dropped_count})")
else:
    pos_weight = CONFIG["pos_weight"]
    print(f"Manual pos_weight: {pos_weight}")

# CrossEntropy with class weights: [weight_dropped, weight_survived]
class_weights = torch.tensor([1.0, float(pos_weight)], dtype=torch.float).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights: {class_weights.tolist()}")

## 7. Training loop

In [ ]:
def train_epoch(model, tiles, optimizer, criterion, config, device):
    """
    One training epoch: iterate over all tiles, build a graph per tile,
    forward pass, compute loss, backprop.
    Returns mean loss and mean accuracy over all nodes.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    # Shuffle tile order each epoch
    random.shuffle(tiles)

    for tile in tiles:
        bgt = gpd.read_file(tile["bgt"])
        if "survived" not in bgt.columns or len(bgt) < 2:
            continue

        graph = tile_to_graph(bgt, config).to(device)

        optimizer.zero_grad()
        logits = model(graph.x, graph.edge_index)  # (N, 2)
        loss   = criterion(logits, graph.y)
        loss.backward()
        optimizer.step()

        preds    = logits.argmax(dim=1)
        correct += (preds == graph.y).sum().item()
        total   += graph.y.shape[0]
        total_loss += loss.item()

    return total_loss / max(len(tiles), 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(model, tiles, criterion, config, device):
    """
    Evaluate on a set of tiles.
    Returns loss, accuracy, precision, recall, F1 over all nodes.
    """
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for tile in tiles:
        bgt = gpd.read_file(tile["bgt"])
        if "survived" not in bgt.columns or len(bgt) < 2:
            continue

        graph  = tile_to_graph(bgt, config).to(device)
        logits = model(graph.x, graph.edge_index)
        loss   = criterion(logits, graph.y)
        total_loss += loss.item()

        preds = logits.argmax(dim=1).cpu().numpy()
        labels = graph.y.cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels)

    all_preds  = np.concatenate(all_preds)  if all_preds  else np.array([])
    all_labels = np.concatenate(all_labels) if all_labels else np.array([])

    if len(all_preds) == 0:
        return total_loss, 0.0, 0.0, 0.0, 0.0

    acc = (all_preds == all_labels).mean()

    # Precision, recall, F1 for the 'survived' class (label=1)
    tp = ((all_preds == 1) & (all_labels == 1)).sum()
    fp = ((all_preds == 1) & (all_labels == 0)).sum()
    fn = ((all_preds == 0) & (all_labels == 1)).sum()

    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-8)

    return total_loss / max(len(tiles), 1), acc, precision, recall, f1


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5, verbose=True
)

print("Training setup ready.")

In [ ]:
train_losses, val_losses, val_f1s = [], [], []
best_val_loss      = float("inf")
epochs_no_improve  = 0
best_model_path    = CONFIG["model_dir"] / "best_graphsage.pt"

train_tiles = tile_index["train"]
val_tiles   = tile_index["val"]

print(f"Training on {len(train_tiles)} tiles, validating on {len(val_tiles)} tiles")
print(f"Epochs: {CONFIG['num_epochs']}  |  Device: {DEVICE}")
print("─" * 70)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    train_loss, train_acc = train_epoch(
        model, train_tiles, optimizer, criterion, CONFIG, DEVICE
    )
    val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(
        model, val_tiles, criterion, CONFIG, DEVICE
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        flag = "  ← saved"
    else:
        epochs_no_improve += 1
        flag = ""

    print(
        f"Epoch {epoch:3d}/{CONFIG['num_epochs']} │ "
        f"train {train_loss:.4f} acc {train_acc*100:.1f}% │ "
        f"val {val_loss:.4f} acc {val_acc*100:.1f}% "
        f"F1 {val_f1:.3f}"
        f"{flag}"
    )

    if epochs_no_improve >= CONFIG["early_stop_patience"]:
        print(f"\nEarly stop at epoch {epoch}.")
        break

print(f"\nBest val loss: {best_val_loss:.4f}")
print(f"Model saved to: {best_model_path}")

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label="Train", color="steelblue")
axes[0].plot(val_losses,   label="Val",   color="tomato")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(val_f1s, color="seagreen")
axes[1].set_title("Validation F1 (survived class)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG["model_dir"] / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved.")

## 9. Visualize predictions on one val tile

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

# Pick first val tile
val_tile = tile_index["val"][0]
bgt_val  = gpd.read_file(val_tile["bgt"])
brt_val  = gpd.read_file(val_tile["brt"])

graph_val = tile_to_graph(bgt_val, CONFIG).to(DEVICE)
with torch.no_grad():
    logits_val = model(graph_val.x, graph_val.edge_index)
    pred_labels = logits_val.argmax(dim=1).cpu().numpy()

bgt_val["pred_survived"] = pred_labels

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# True labels
if "survived" in bgt_val.columns:
    bgt_val[bgt_val["survived"] == 1].plot(ax=axes[0], color="steelblue", alpha=0.7)
    bgt_val[bgt_val["survived"] == 0].plot(ax=axes[0], color="tomato",    alpha=0.5)
    axes[0].set_title("True labels (blue=survived, red=dropped)")
else:
    bgt_val.plot(ax=axes[0], color="grey")
    axes[0].set_title("BGT (no labels)")

# Predicted labels
bgt_val[bgt_val["pred_survived"] == 1].plot(ax=axes[1], color="steelblue", alpha=0.7)
bgt_val[bgt_val["pred_survived"] == 0].plot(ax=axes[1], color="tomato",    alpha=0.5)
axes[1].set_title("Predicted labels (blue=survived, red=dropped)")

# BRT target
brt_val.plot(ax=axes[2], color="seagreen", edgecolor="white", linewidth=0.3)
axes[2].set_title(f"BRT target ({len(brt_val)} polygons)")

for ax in axes:
    ax.set_axis_off()

plt.suptitle("Val tile: true vs predicted survival", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["model_dir"] / "val_predictions.png", dpi=150, bbox_inches="tight")
plt.show()